# Steering-vector extraction — atomic cognitive mechanisms

Phase 1 (**vLLM**): generate persona-conditioned completions over a neutral prompt bank.  
Phase 2 (**repeng**, HF transformers): build ON/baseline contrastive pairs and train one control
vector per mechanism (PCA on last-token hidden-state differences).

Mechanisms & personas live in `deep/steering/personas.py` (mirrors `context/pilot_personas.md`).

**Generations are saved to Google Drive**, so if the runtime breaks you can restart, re-run the
setup cells, and jump straight to Phase 2 — it reloads the dataset from Drive. The vLLM engine is
explicitly ejected before Phase 2 re-loads the model in HF transformers (vLLM can't expose hidden
states).

## 1. Setup

In [ ]:
import os, subprocess, sys
from pathlib import Path

REPO_URL = "https://github.com/ChuloIva/RL_rh"
REPO_DIR = Path.cwd() / "RL_rh"
if (Path.cwd() / "deep" / "runner.py").exists():
    REPO_DIR = Path.cwd()                      # notebook already inside the repo
elif not REPO_DIR.exists():
    subprocess.check_call(["git", "clone", REPO_URL, str(REPO_DIR)])
os.chdir(REPO_DIR)
sys.path.insert(0, str(REPO_DIR / "deep"))    # so `from steering import ...` works
print("cwd:", os.getcwd())

In [ ]:
%pip install -q vllm repeng transformers accelerate
# repeng is pure-Python (PCA + HF hooks) and reuses the torch/transformers that vLLM installs.
# If Colab's CUDA doesn't match vLLM's default wheel, pin a known-good version, e.g.:
#   %pip install -q "vllm==0.6.3"

In [ ]:
# API keys — only HF_TOKEN matters here (and only for gated models; Qwen2.5 is public).
import os
from pathlib import Path
_KEYS = ["HF_TOKEN"]
try:
    from google.colab import userdata  # type: ignore
    for k in _KEYS:
        if not os.environ.get(k):
            try:
                v = userdata.get(k)
                if v: os.environ[k] = v
            except Exception: pass
except ImportError:
    pass
try:
    from dotenv import load_dotenv
    for p in [REPO_DIR / ".env", Path.cwd() / ".env"]:
        if p.exists(): load_dotenv(p, override=False)
except ImportError:
    pass
print("HF_TOKEN:", "set" if os.environ.get("HF_TOKEN") else "not set (fine for public models)")

### Mount Drive & define persistent paths
Generations + trained vectors are written to Drive so they survive a runtime restart.

In [ ]:
from pathlib import Path
try:
    from google.colab import drive  # type: ignore
    drive.mount("/content/drive")
    OUT_DIR = Path("/content/drive/MyDrive/steering_pilot")
except Exception as e:
    print("Drive not mounted (", e, ") — falling back to local dir")
    OUT_DIR = REPO_DIR / "deep" / "steering" / "out"
OUT_DIR.mkdir(parents=True, exist_ok=True)

GEN_PATH     = str(OUT_DIR / "generations.jsonl")
VECTORS_PATH = str(OUT_DIR / "control_vectors.pkl")
META_PATH    = str(OUT_DIR / "metadata.json")
print("artifacts ->", OUT_DIR)
print("generations exist already?", Path(GEN_PATH).exists())

## 2. Phase 1 — generate data (vLLM)

For each of the 10 mechanisms × {on, baseline} × 50 neutral prompts, generate a completion and
write to `GEN_PATH` on Drive.

**Skip this whole section** (Phase 1) if `generations.jsonl` already exists on Drive — go straight
to Phase 2.

In [ ]:
from vllm import LLM
from steering import config, generate

gen_cfg = config.GenConfig()
# gen_cfg.n_samples = 2; gen_cfg.max_tokens = 256   # tweak if you want more / longer data

# Build the engine explicitly so we can eject it cleanly before Phase 2.
llm = LLM(
    model=gen_cfg.model_name,
    dtype=gen_cfg.dtype,
    max_model_len=gen_cfg.max_model_len,
    gpu_memory_utilization=gen_cfg.gpu_memory_utilization,
)
records = generate.generate_dataset(gen_cfg, out_path=GEN_PATH, llm=llm)
print(len(records), "records ->", GEN_PATH)

In [ ]:
# Tier-0 eyeball: does the ON persona actually exhibit the mechanism vs its matched baseline?
def peek(mech, prompt_idx=0, n=500):
    for variant in ("on", "baseline"):
        r = next((x for x in records if x["mechanism"]==mech
                  and x["variant"]==variant and x["prompt_idx"]==prompt_idx), None)
        if r:
            print(f"\n[{mech} / {variant}]  prompt: {r['prompt']}\n{r['completion'][:n]}")

peek("rumination")
peek("threat_vigilance")

### Eject the vLLM engine
Free all GPU memory held by vLLM before Phase 2 re-loads the model in HF transformers.

In [ ]:
import gc, torch
try:
    del llm                       # drop our reference to the vLLM engine
except NameError:
    pass
try:
    from vllm.distributed.parallel_state import (destroy_model_parallel,
                                                  destroy_distributed_environment)
    destroy_model_parallel(); destroy_distributed_environment()
except Exception as e:
    print("vllm teardown skipped:", e)
gc.collect(); torch.cuda.empty_cache()
print("GPU allocated:", round(torch.cuda.memory_allocated()/1e9, 2), "GB")
# If GPU memory isn't fully freed (vLLM can be sticky): Runtime -> Restart, re-run cells 1-6,
# then skip Phase 1 and run Phase 2 — it reloads generations from Drive.

## 3. Phase 2 — extract control vectors (repeng)

Loads the model fresh in HF transformers (needed for hidden states), builds contrastive pairs from
the Drive-saved generations, and trains one `ControlVector` per mechanism.

This section is self-contained: after a runtime restart, run cells 1–6 then come straight here.

In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer
from steering import config

ecfg = config.ExtractConfig()
# Optional: restrict to a middle layer band where style/persona usually steers cleanest:
# ecfg.hidden_layers = list(range(-6, -19, -1))

tok = AutoTokenizer.from_pretrained(ecfg.model_name)
if tok.pad_token_id is None:
    tok.pad_token = tok.eos_token
model = AutoModelForCausalLM.from_pretrained(
    ecfg.model_name, torch_dtype=torch.bfloat16, device_map="cuda",
)
model.eval()
print("layers:", model.config.num_hidden_layers, "| model_type:", model.config.model_type)

In [ ]:
from steering import extract, config

records = extract.load_records(GEN_PATH)            # reload from Drive (robust to restart)
print("loaded", len(records), "records from", GEN_PATH)
pairs = extract.build_pairs(records, tok, ecfg)
print("pairs per mechanism:", {m: len(p) for m, p in pairs.items()})

vectors = extract.extract_vectors(model, tok, pairs, ecfg)

In [ ]:
# Separability check: cosine-similarity matrix between mechanism directions at a mid layer.
any_v = next(iter(vectors.values()))
layers = sorted(any_v.directions.keys())
layer = layers[len(layers) // 2]
labels, mat = extract.cosine_matrix(vectors, layer)
print(f"cosine similarity @ layer {layer}\n")
print("              " + "".join(f"{l[:8]:>9s}" for l in labels))
for i, l in enumerate(labels):
    print(f"{l[:13]:13s} " + "".join(f"{mat[i,j]:+9.2f}" for j in range(len(labels))))

In [ ]:
# Save bundle to Drive (plain-dict pickle, loadable without repeng) + metadata.json, then download.
from steering import extract
extract.save_bundle(vectors, VECTORS_PATH, model_name=ecfg.model_name, cfg=ecfg,
                    pairs=pairs, meta_path=META_PATH)
try:
    from google.colab import files  # type: ignore
    files.download(VECTORS_PATH)
    files.download(META_PATH)
except ImportError:
    print("Not on Colab — vectors at", VECTORS_PATH)
# (Also safe on Drive at VECTORS_PATH even if the download dialog is skipped.)

## Next: steering notebook

Load the bundle locally and steer with repeng's `ControlModel`:
```python
from steering import extract
bundle = extract.load_bundle("control_vectors.pkl")
vecs = extract.bundle_to_control_vectors(bundle)   # {mechanism: ControlVector}
# from repeng import ControlModel; wrap model, model.set_control(vecs['rumination'], coeff); ...
# Compose: vecs['rumination'] + vecs['negative_self_schema'] + vecs['hopelessness']  # -> depression
```
Composition recipes are in `personas.SYNDROME_RECIPES` / `personas.COMPOUND_PREDICTIONS`.